#CÀI ĐẶT

In [99]:
import pandas as pd
import numpy as np
import warnings
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from prophet import Prophet
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.linear_model import HuberRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error

SEED         = 6001
TUNE_TRIALS  = 40
np.random.seed(SEED)

# 1.  DATA LOAD

In [100]:

df_sales  = (pd.read_csv('../dataset/sales.csv',             parse_dates=['Date'])
               .sort_values('Date').reset_index(drop=True))
df_sub    = (pd.read_csv('../dataset/sample_submission.csv', parse_dates=['Date'])
               .sort_values('Date').reset_index(drop=True))
df_promos = pd.read_csv('../dataset/promotions.csv', parse_dates=['start_date', 'end_date'])

print(f"Train : {df_sales.shape}  | {df_sales.Date.min().date()} → {df_sales.Date.max().date()}")
print(f"Test  : {df_sub.shape}    | {df_sub.Date.min().date()} → {df_sub.Date.max().date()}")

df_sales['Revenue_log'] = np.log1p(df_sales['Revenue'])
df_sales['COGS_Ratio']  = df_sales['COGS'] / (df_sales['Revenue'] + 1e-9)
ORIGIN_DATE             = df_sales['Date'].min()

Train : (3833, 3)  | 2012-07-04 → 2022-12-31
Test  : (548, 3)    | 2023-01-01 → 2024-07-01


# 2. FEATURE ENGINEERING


1.  PROMO PROBABILITY — tần suất lịch sử dựa trên cặp (month, day).
2.  CAMPAIGN WINDOW FACTORY — bộ tạo cửa sổ chiến dịch.
3.  CAMPAIGN FEATURES — vector hóa + multi-scale decay pulses.
4.  CAMPAIGN DENSITY — look-ahead coverage, sử dụng O(N) cumsum.
5.  REVENUE APEX DISTANCE




In [101]:

# ─────────────────────────────────────────────────────────────────────────────
# PROMO PROBABILITY
# ─────────────────────────────────────────────────────────────────────────────
_pdays = []
for _, row in df_promos.iterrows():
    _pdays.extend(pd.date_range(row['start_date'], row['end_date']))
_pdf              = pd.DataFrame({'Date': _pdays})
_pdf['month']     = _pdf['Date'].dt.month
_pdf['day']       = _pdf['Date'].dt.day
promo_counts      = _pdf.groupby(['month', 'day']).size().reset_index(name='promo_years')
promo_counts['promo_prob'] = (promo_counts['promo_years'] / 10.0).clip(upper=1.0)


# ─────────────────────────────────────────────────────────────────────────────
# CAMPAIGN WINDOW FACTORY
# ─────────────────────────────────────────────────────────────────────────────
def _build_promo_windows(years):
    CAMPAIGN_CALENDAR = [
        ( 1, 30, 30, 1),
        ( 3, 18, 30, 2),
        ( 6, 23, 29, 3),
        ( 7, 30, 34, 5),
        ( 8, 30, 32, 4),
        (11, 18, 45, 6),
    ]
    windows = []
    for yr in years:
        for mo, sd, dur, rank in CAMPAIGN_CALENDAR:
            try:
                ws = pd.Timestamp(year=yr, month=mo, day=sd)
            except ValueError:
                ws = pd.Timestamp(year=yr, month=mo, day=1) + pd.offsets.MonthEnd(0)
            we = ws + pd.Timedelta(days=dur - 1)
            windows.append((ws, we, rank))
    return windows


# ─────────────────────────────────────────────────────────────────────────────
# CAMPAIGN FEATURES
# ─────────────────────────────────────────────────────────────────────────────
def attach_campaign_features(df: pd.DataFrame) -> pd.DataFrame:

    df      = df.copy()
    years   = sorted(set(df['Date'].dt.year.tolist()))
    windows = _build_promo_windows(years)
    dates_d = df['Date'].values.astype('datetime64[D]')

    n_flag  = np.zeros(len(df), dtype=np.int8)
    n_rank  = np.zeros(len(df), dtype=np.int8)
    n_prog  = np.zeros(len(df), dtype=np.float32)
    d2next  = np.full(len(df), 999.0)
    d2prev  = np.full(len(df), 999.0)

    for ws, we, rank in windows:
        ws_d = np.datetime64(ws.date(), 'D')
        we_d = np.datetime64(we.date(), 'D')

        m_in     = (dates_d >= ws_d) & (dates_d <= we_d)
        m_before = dates_d  < ws_d
        m_after  = dates_d  > we_d

        n_flag[m_in] = 1
        n_rank[m_in] = rank
        dur          = (we - ws).days + 1
        n_prog[m_in] = (dates_d[m_in] - ws_d).astype('int') / dur

        d2next[m_before] = np.minimum(
            d2next[m_before], (ws_d - dates_d[m_before]).astype('int'))
        d2prev[m_after]  = np.minimum(
            d2prev[m_after],  (dates_d[m_after] - we_d).astype('int'))

    inside         = n_flag == 1
    d2next[inside] = 0.0
    d2prev[inside] = 0.0

    df['in_campaign']         = n_flag.astype(int)
    df['campaign_rank']       = n_rank.astype(int)
    df['campaign_progress']   = n_prog
    df['days_to_campaign']    = d2next
    df['days_post_campaign']  = d2prev
    df['pre_campaign_pulse']  = np.exp(-d2next / 14.0)
    df['post_campaign_trail'] = np.exp(-d2prev / 14.0)
    df['pre_campaign_sharp']  = np.exp(-d2next /  7.0)
    df['post_campaign_long']  = np.exp(-d2prev / 30.0)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# CAMPAIGN DENSITY
# ─────────────────────────────────────────────────────────────────────────────
def compute_campaign_density(dates: pd.Series) -> np.ndarray:
    """
    For each date d returns the fraction of days in (d+1 … d+30) that
    fall inside any campaign window.

    Implementation: build a binary indicator array over the full calendar
    span, take its prefix sum, then answer each query in O(1) via two
    index lookups — no Python-level loop over dates.
    """
    all_years = sorted(set(dates.dt.year.tolist() + (dates.dt.year + 1).tolist()))
    windows   = _build_promo_windows(all_years)

    cal_start = dates.min()
    cal_end   = dates.max() + pd.Timedelta(days=32)
    cal_arr   = pd.date_range(cal_start, cal_end, freq='D').values.astype('datetime64[D]')

    flag = np.zeros(len(cal_arr), dtype=np.float32)
    for ws, we, _ in windows:
        lo = int(np.searchsorted(cal_arr, np.datetime64(ws.date(), 'D')))
        hi = int(np.searchsorted(cal_arr, np.datetime64(we.date(), 'D'), side='right'))
        flag[lo:hi] = 1.0

    cum     = np.concatenate([[0.0], np.cumsum(flag)])
    date_d  = dates.values.astype('datetime64[D]')
    idx     = np.searchsorted(cal_arr, date_d)
    lo_idx  = idx + 1
    hi_idx  = np.minimum(idx + 31, len(cum) - 1)
    return (cum[hi_idx] - cum[lo_idx]) / 30.0


# ─────────────────────────────────────────────────────────────────────────────
# REVENUE APEX DISTANCE
# ─────────────────────────────────────────────────────────────────────────────
_APEX_MONTHS = [4, 5, 11]

def dist_to_revenue_apex(dates: pd.Series) -> np.ndarray:
    """
    For each date returns the minimum calendar-day distance to the nearest
    known revenue-peak anchor (1st of Apr, May, Nov for years ±1).

    Uses numpy broadcasting over the full date series × all anchors —
    single matrix operation, result in O(N·K) with K ≈ n_years × 9.
    """
    unique_years = sorted(set(dates.dt.year.tolist()))
    apex_ts      = pd.DatetimeIndex([
        pd.Timestamp(yr + dy, m, 1)
        for yr in unique_years for dy in [-1, 0, 1]
        for m  in _APEX_MONTHS
    ])
    apex_ns  = apex_ts.asi8
    date_ns  = dates.values.astype('datetime64[ns]').astype('int64')
    diff_day = np.abs(date_ns[:, None] - apex_ns[None, :]) / 86_400_000_000_000
    return diff_day.min(axis=1)




#3. PROPHET VÀ CÁC FEATURE LIÊN QUAN


In [102]:
# ─────────────────────────────────────────────────────────────────────────────
# PROPHET  — seasonality baseline
# ─────────────────────────────────────────────────────────────────────────────
def fit_seasonal_prophet(dates, log_revenues):
    """
    Fit a Prophet model on log-revenue using yearly + weekly seasonality.
    No VN holiday list injected — Prophet learns the seasonal shape from data.
    Returns a fitted Prophet object.
    """
    m = Prophet(
        yearly_seasonality       = True,
        weekly_seasonality       = True,
        daily_seasonality        = False,
        changepoint_prior_scale  = 0.05,
        seasonality_prior_scale  = 10.0,
        changepoint_range        = 0.95,
        seasonality_mode         = 'additive',
    )
    m.fit(pd.DataFrame({'ds': dates, 'y': log_revenues}))
    return m


def extract_prophet_components(model, dates):
    fc = model.predict(pd.DataFrame({'ds': dates}))
    return {
        'prophet_baseline': fc['yhat'].values,
        'prophet_trend'   : fc['trend'].values,
        'prophet_weekly'  : fc['weekly'].values,
        'prophet_yearly'  : fc['yearly'].values,
        'prophet_uncert'  : (fc['yhat_upper'] - fc['yhat_lower']).values,
    }


# ─────────────────────────────────────────────────────────────────────────────
# STATIC FEATURE FRAME
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 2 — Build static feature frames")
print("=" * 65)

def build_feature_frame(df):
    """
    Construct the date-indexed feature matrix from calendar arithmetic,
    campaign windows, and promo-probability tables only.
    Prophet components are merged into this frame inside the CV loop.
    """
    out = df[['Date']].copy().sort_values('Date').reset_index(drop=True)
    yr  = out['Date'].dt.year
    mo  = out['Date'].dt.month
    dy  = out['Date'].dt.day
    dow = out['Date'].dt.dayofweek
    dim = out['Date'].dt.days_in_month

    out['year']           = yr
    out['month']          = mo
    out['day']            = dy
    out['dow']            = dow
    out['is_weekend']     = (dow >= 5).astype(int)
    out['is_monthend']    = out['Date'].dt.is_month_end.astype(int)
    out['is_monthstart']  = out['Date'].dt.is_month_start.astype(int)
    out['t']              = (out['Date'] - ORIGIN_DATE).dt.days
    out['doy_norm']       = out['Date'].dt.dayofyear / 365.0
    out['month_progress'] = dy / dim
    out['year_parity']    = (yr % 2).astype(int)
    out['month_sin']      = np.sin(2 * np.pi * mo / 12)
    out['month_cos']      = np.cos(2 * np.pi * mo / 12)

    out['is_peak_season'] = mo.isin([4, 5, 11]).astype(int)
    out['peak_proximity'] = 1.0 / (1.0 + dist_to_revenue_apex(out['Date']))

    camp = attach_campaign_features(out[['Date']])
    for col in ['in_campaign', 'campaign_rank', 'campaign_progress',
                'days_to_campaign', 'days_post_campaign',
                'pre_campaign_pulse', 'post_campaign_trail',
                'pre_campaign_sharp', 'post_campaign_long']:
        out[col] = camp[col].values

    print(f"  computing campaign_density_30 for {len(out)} rows …")
    out['campaign_density_30'] = compute_campaign_density(out['Date'])

    out = out.merge(promo_counts[['month', 'day', 'promo_prob']],
                    on=['month', 'day'], how='left')
    out['promo_prob'] = out['promo_prob'].fillna(0)

    return out.sort_values('Date').reset_index(drop=True)


print("Building train feature frame …")
train_static = build_feature_frame(df_sales)
print("Building test feature frame …")
test_static  = build_feature_frame(df_sub)

STATIC_COLS   = [c for c in train_static.columns if c != 'Date']
PROPHET_COLS  = ['prophet_trend', 'prophet_weekly', 'prophet_yearly', 'prophet_uncert']
INTERACT_COLS = ['campaign_x_peak', 'promo_x_campaign', 'uncert_x_campaign']
ALL_FEAT_COLS = STATIC_COLS + PROPHET_COLS + INTERACT_COLS

print(f"\nStatic   : {len(STATIC_COLS)}")
print(f"Prophet  : {len(PROPHET_COLS)}")
print(f"Interact : {len(INTERACT_COLS)}")
print(f"Total    : {len(ALL_FEAT_COLS)}")
print(f"Columns  : {ALL_FEAT_COLS}")


# ─────────────────────────────────────────────────────────────────────────────
# INTERACTION TERMS
# ─────────────────────────────────────────────────────────────────────────────
def inject_interactions(df):
    """
    Compute multiplicative cross-terms that require Prophet columns.
    Called inside the CV loop and during full retrain, never before.
    """
    df = df.copy()
    df['campaign_x_peak']   = df['campaign_rank']  * df['is_peak_season']
    df['promo_x_campaign']  = df['promo_prob']      * df['in_campaign']
    df['uncert_x_campaign'] = df['prophet_uncert']  * df['in_campaign']
    return df




STEP 2 — Build static feature frames
Building train feature frame …
  computing campaign_density_30 for 3833 rows …
Building test feature frame …
  computing campaign_density_30 for 548 rows …

Static   : 26
Prophet  : 4
Interact : 3
Total    : 33
Columns  : ['year', 'month', 'day', 'dow', 'is_weekend', 'is_monthend', 'is_monthstart', 't', 'doy_norm', 'month_progress', 'year_parity', 'month_sin', 'month_cos', 'is_peak_season', 'peak_proximity', 'in_campaign', 'campaign_rank', 'campaign_progress', 'days_to_campaign', 'days_post_campaign', 'pre_campaign_pulse', 'post_campaign_trail', 'pre_campaign_sharp', 'post_campaign_long', 'campaign_density_30', 'promo_prob', 'prophet_trend', 'prophet_weekly', 'prophet_yearly', 'prophet_uncert', 'campaign_x_peak', 'promo_x_campaign', 'uncert_x_campaign']


#4. OPTUNA TUNING CÁC BOOSTER

In [103]:
def _tune_xgb(X_tr, y_tr, X_val, y_val, n_trials):
    def _obj(trial):
        p = dict(
            n_estimators     = trial.suggest_int('n_estimators', 500, 3000),
            learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            max_depth        = trial.suggest_int('max_depth', 3, 6),
            subsample        = trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            min_child_weight = trial.suggest_int('min_child_weight', 1, 20),
            objective='reg:pseudohubererror', tree_method='hist', random_state=SEED,
        )
        m = xgb.XGBRegressor(**p, early_stopping_rounds=50)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        return mean_absolute_error(y_val, m.predict(X_val))

    study = optuna.create_study(
        direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(_obj, n_trials=n_trials, show_progress_bar=False)
    return {**study.best_params,
            'objective': 'reg:pseudohubererror', 'tree_method': 'hist', 'random_state': SEED}


def _tune_lgb(X_tr, y_tr, X_val, y_val, n_trials):
    """Search LightGBM hyperparameters; objective fixed to Huber."""
    def _obj(trial):
        p = dict(
            n_estimators      = trial.suggest_int('n_estimators', 500, 3000),
            learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
            max_depth         = trial.suggest_int('max_depth', 3, 6),
            num_leaves        = trial.suggest_int('num_leaves', 16, 63),
            subsample         = trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            min_child_samples = trial.suggest_int('min_child_samples', 5, 50),
            objective='huber', random_state=SEED, verbose=-1,
        )
        m = lgb.LGBMRegressor(**p)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        return mean_absolute_error(y_val, m.predict(X_val))

    study = optuna.create_study(
        direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(_obj, n_trials=n_trials, show_progress_bar=False)
    return {**study.best_params, 'objective': 'huber', 'random_state': SEED, 'verbose': -1}


def _tune_cat(X_tr, y_tr, X_val, y_val, n_trials):
    def _obj(trial):
        p = dict(
            iterations    = trial.suggest_int('iterations', 500, 3000),
            learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            depth         = trial.suggest_int('depth', 3, 7),
            l2_leaf_reg   = trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
            loss_function='Huber:delta=1.5', random_state=SEED, verbose=False,
        )
        m = CatBoostRegressor(**p, early_stopping_rounds=50)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        return mean_absolute_error(y_val, m.predict(X_val))

    study = optuna.create_study(
        direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(_obj, n_trials=n_trials, show_progress_bar=False)
    return {**study.best_params,
            'loss_function': 'Huber:delta=1.5', 'random_state': SEED, 'verbose': False}


#5. WALK-FORWARD CV

In [104]:

FOLDS = [
    ('2012-07-04', '2018-12-31', '2019-01-01', '2019-12-31'),
    ('2012-07-04', '2019-12-31', '2020-01-01', '2020-12-31'),
    ('2012-07-04', '2020-12-31', '2021-01-01', '2021-12-31'),
    ('2012-07-04', '2021-12-31', '2022-01-01', '2022-12-31'),
]


def _assemble_matrix(static_df, dates_df, comp):
    sf = static_df[static_df['Date'].isin(dates_df['Date'])].copy()
    sf = sf.set_index('Date').loc[dates_df['Date'].values].reset_index()
    for col in PROPHET_COLS:
        sf[col] = comp[col]
    sf = inject_interactions(sf)
    sf[ALL_FEAT_COLS] = sf[ALL_FEAT_COLS].fillna(-1)
    return sf[ALL_FEAT_COLS].values


# ── Fold-1 Optuna tuning  ──────────────────────────────
print("\nTuning hyperparameters on fold-1 …")
_tr1_s, _tr1_e, _va1_s, _va1_e = FOLDS[0]
_df_tr1 = df_sales[(df_sales['Date'] >= _tr1_s) & (df_sales['Date'] <= _tr1_e)].reset_index(drop=True)
_df_va1 = df_sales[(df_sales['Date'] >= _va1_s) & (df_sales['Date'] <= _va1_e)].reset_index(drop=True)

_proph1    = fit_seasonal_prophet(_df_tr1['Date'], _df_tr1['Revenue_log'])
_ctr1      = extract_prophet_components(_proph1, _df_tr1['Date'])
_cva1      = extract_prophet_components(_proph1, _df_va1['Date'])
_res_tr1   = _df_tr1['Revenue_log'].values - _ctr1['prophet_baseline']
_res_va1   = _df_va1['Revenue_log'].values - _cva1['prophet_baseline']
_X_tr1     = _assemble_matrix(train_static, _df_tr1, _ctr1)
_X_va1     = _assemble_matrix(train_static, _df_va1, _cva1)

if TUNE_TRIALS > 0:
    print(f"  XGB  ({TUNE_TRIALS} trials) …")
    XGB_PARAMS = _tune_xgb(_X_tr1, _res_tr1, _X_va1, _res_va1, TUNE_TRIALS)
    print(f"  LGB  ({TUNE_TRIALS} trials) …")
    LGB_PARAMS = _tune_lgb(_X_tr1, _res_tr1, _X_va1, _res_va1, TUNE_TRIALS)
    print(f"  CAT  ({TUNE_TRIALS} trials) …")
    CAT_PARAMS = _tune_cat(_X_tr1, _res_tr1, _X_va1, _res_va1, TUNE_TRIALS)
    print(f"\n  XGB best : {XGB_PARAMS}")
    print(f"  LGB best : {LGB_PARAMS}")
    print(f"  CAT best : {CAT_PARAMS}")
else:
    XGB_PARAMS = {
        'n_estimators': 2000, 'learning_rate': 0.077, 'max_depth': 4,
        'objective': 'reg:pseudohubererror', 'tree_method': 'hist',
        'subsample': 0.964, 'colsample_bytree': 0.999, 'min_child_weight': 6,
        'random_state': SEED,
    }
    LGB_PARAMS = {
        'n_estimators': 2000, 'learning_rate': 0.010, 'max_depth': 4,
        'objective': 'huber', 'num_leaves': 28,
        'subsample': 0.933, 'colsample_bytree': 0.853,
        'random_state': SEED, 'verbose': -1,
    }
    CAT_PARAMS = {
        'iterations': 2000, 'learning_rate': 0.099, 'depth': 4,
        'loss_function': 'Huber:delta=1.5', 'l2_leaf_reg': 5.356,
        'random_state': SEED, 'verbose': False,
    }

# ── Full walk-forward CV ──────────────────────────────────────────────────────
oof_rev_xgb, oof_rev_lgb, oof_rev_cat = [], [], []
oof_rat_xgb, oof_rat_lgb, oof_rat_cat = [], [], []
oof_y_rev_log, oof_y_rat               = [], []

for fold_i, (tr_s, tr_e, va_s, va_e) in enumerate(FOLDS):
    print(f"\nFold {fold_i+1}: train {tr_s}→{tr_e} | val {va_s}→{va_e}")
    df_tr  = df_sales[(df_sales['Date'] >= tr_s) & (df_sales['Date'] <= tr_e)].reset_index(drop=True)
    df_val = df_sales[(df_sales['Date'] >= va_s) & (df_sales['Date'] <= va_e)].reset_index(drop=True)

    proph = fit_seasonal_prophet(df_tr['Date'], df_tr['Revenue_log'])
    c_tr  = extract_prophet_components(proph, df_tr['Date'])
    c_val = extract_prophet_components(proph, df_val['Date'])

    res_tr  = df_tr['Revenue_log'].values  - c_tr['prophet_baseline']
    res_val = df_val['Revenue_log'].values - c_val['prophet_baseline']

    X_tr      = _assemble_matrix(train_static, df_tr,  c_tr)
    X_val     = _assemble_matrix(train_static, df_val, c_val)
    rat_tr    = df_tr['COGS_Ratio'].values
    rat_val   = df_val['COGS_Ratio'].values

    # Revenue residual models
    m_xgb = xgb.XGBRegressor(**XGB_PARAMS, early_stopping_rounds=50)
    m_xgb.fit(X_tr, res_tr, eval_set=[(X_val, res_val)], verbose=False)

    m_lgb = lgb.LGBMRegressor(**LGB_PARAMS)
    m_lgb.fit(X_tr, res_tr, eval_set=[(X_val, res_val)],
              callbacks=[lgb.early_stopping(50, verbose=False)])

    m_cat = CatBoostRegressor(**CAT_PARAMS, early_stopping_rounds=50)
    m_cat.fit(X_tr, res_tr, eval_set=[(X_val, res_val)], verbose=False)

    log_p_xgb = c_val['prophet_baseline'] + m_xgb.predict(X_val)
    log_p_lgb = c_val['prophet_baseline'] + m_lgb.predict(X_val)
    log_p_cat = c_val['prophet_baseline'] + m_cat.predict(X_val)

    oof_rev_xgb.extend(log_p_xgb)
    oof_rev_lgb.extend(log_p_lgb)
    oof_rev_cat.extend(log_p_cat)
    oof_y_rev_log.extend(df_val['Revenue_log'].values)

    mae_xgb = mean_absolute_error(df_val['Revenue'], np.expm1(log_p_xgb))
    mae_lgb = mean_absolute_error(df_val['Revenue'], np.expm1(log_p_lgb))
    mae_cat = mean_absolute_error(df_val['Revenue'], np.expm1(log_p_cat))
    print(f"  Revenue MAE → XGB: {mae_xgb:,.0f}  LGB: {mae_lgb:,.0f}  CAT: {mae_cat:,.0f}")

    # COGS ratio models
    r_xgb = xgb.XGBRegressor(**XGB_PARAMS, early_stopping_rounds=50)
    r_xgb.fit(X_tr, rat_tr, eval_set=[(X_val, rat_val)], verbose=False)

    r_lgb = lgb.LGBMRegressor(**LGB_PARAMS)
    r_lgb.fit(X_tr, rat_tr, eval_set=[(X_val, rat_val)],
              callbacks=[lgb.early_stopping(50, verbose=False)])

    r_cat = CatBoostRegressor(**CAT_PARAMS, early_stopping_rounds=50)
    r_cat.fit(X_tr, rat_tr, eval_set=[(X_val, rat_val)], verbose=False)

    oof_rat_xgb.extend(r_xgb.predict(X_val))
    oof_rat_lgb.extend(r_lgb.predict(X_val))
    oof_rat_cat.extend(r_cat.predict(X_val))
    oof_y_rat.extend(rat_val)




Tuning hyperparameters on fold-1 …


23:50:29 - cmdstanpy - INFO - Chain [1] start processing
23:50:29 - cmdstanpy - INFO - Chain [1] done processing


  XGB  (40 trials) …
  LGB  (40 trials) …
  CAT  (40 trials) …

  XGB best : {'n_estimators': 1073, 'learning_rate': 0.06327688424339933, 'max_depth': 3, 'subsample': 0.9652373281243151, 'colsample_bytree': 0.9253163936070884, 'min_child_weight': 9, 'objective': 'reg:pseudohubererror', 'tree_method': 'hist', 'random_state': 6001}
  LGB best : {'n_estimators': 1513, 'learning_rate': 0.0388469119597765, 'max_depth': 3, 'num_leaves': 32, 'subsample': 0.8083525370324005, 'colsample_bytree': 0.62796394024827, 'min_child_samples': 35, 'objective': 'huber', 'random_state': 6001, 'verbose': -1}
  CAT best : {'iterations': 626, 'learning_rate': 0.16074199263708533, 'depth': 4, 'l2_leaf_reg': 4.237689470329084, 'loss_function': 'Huber:delta=1.5', 'random_state': 6001, 'verbose': False}

Fold 1: train 2012-07-04→2018-12-31 | val 2019-01-01→2019-12-31


23:50:53 - cmdstanpy - INFO - Chain [1] start processing
23:50:54 - cmdstanpy - INFO - Chain [1] done processing


  Revenue MAE → XGB: 905,475  LGB: 912,677  CAT: 910,873

Fold 2: train 2012-07-04→2019-12-31 | val 2020-01-01→2020-12-31


...

  Revenue MAE → XGB: 692,669  LGB: 738,965  CAT: 670,418

Fold 3: train 2012-07-04→2020-12-31 | val 2021-01-01→2021-12-31


23:51:01 - cmdstanpy - INFO - Chain [1] start processing
23:51:01 - cmdstanpy - INFO - Chain [1] done processing


  Revenue MAE → XGB: 564,081  LGB: 577,990  CAT: 522,774

Fold 4: train 2012-07-04→2021-12-31 | val 2022-01-01→2022-12-31


23:51:03 - cmdstanpy - INFO - Chain [1] start processing
23:51:04 - cmdstanpy - INFO - Chain [1] done processing


  Revenue MAE → XGB: 684,305  LGB: 743,019  CAT: 694,719


#6. ENSEMBLE BLENDER

In [105]:

X_blend_rev = np.column_stack([oof_rev_xgb, oof_rev_lgb, oof_rev_cat])
_rev_grid   = {'epsilon': [1.2, 1.25, 1.35, 1.4, 1.45, 1.5],
               'alpha'  : [1e-4, 1e-3, 1e-2, 0.1]}
blend_rev   = (GridSearchCV(HuberRegressor(fit_intercept=False), _rev_grid,
                             cv=5, scoring='neg_mean_absolute_error')
               .fit(X_blend_rev, oof_y_rev_log).best_estimator_)
print(f"Revenue blend weights (XGB/LGB/CAT): {blend_rev.coef_.round(4)}")

X_blend_rat = np.column_stack([oof_rat_xgb, oof_rat_lgb, oof_rat_cat])
_rat_grid   = {'epsilon': [1.01, 1.05, 1.1, 1.2],
               'alpha'  : [1e-4, 1e-3, 1e-2, 0.1]}
blend_rat   = (GridSearchCV(HuberRegressor(fit_intercept=False), _rat_grid,
                             cv=5, scoring='neg_mean_absolute_error')
               .fit(X_blend_rat, oof_y_rat).best_estimator_)
print(f"Ratio  blend weights (XGB/LGB/CAT): {blend_rat.coef_.round(4)}")

oof_blended = blend_rev.predict(X_blend_rev)
print(f"OOF Revenue MAE blended: "
      f"{mean_absolute_error(np.expm1(oof_y_rev_log), np.expm1(oof_blended)):,.0f}")

Revenue blend weights (XGB/LGB/CAT): [-0.0165 -0.0071  1.0345]
Ratio  blend weights (XGB/LGB/CAT): [-0.1644  0.8621  0.3238]
OOF Revenue MAE blended: 604,144


7. FULL RETRAIN

In [106]:
full_prophet = fit_seasonal_prophet(df_sales['Date'], df_sales['Revenue_log'])
comp_full    = extract_prophet_components(full_prophet, df_sales['Date'])
resid_full   = df_sales['Revenue_log'].values - comp_full['prophet_baseline']


def _assemble_full_matrix(static_df, comp):
    sf = static_df.copy()
    for col in PROPHET_COLS:
        sf[col] = comp[col]
    sf = inject_interactions(sf)
    sf[ALL_FEAT_COLS] = sf[ALL_FEAT_COLS].fillna(-1)
    return sf[ALL_FEAT_COLS].values


X_full     = _assemble_full_matrix(train_static, comp_full)
rat_full   = df_sales['COGS_Ratio'].values

f_rev_xgb = xgb.XGBRegressor(**XGB_PARAMS);  f_rev_xgb.fit(X_full, resid_full, verbose=False)
f_rev_lgb = lgb.LGBMRegressor(**LGB_PARAMS);  f_rev_lgb.fit(X_full, resid_full)
f_rev_cat = CatBoostRegressor(**CAT_PARAMS);  f_rev_cat.fit(X_full, resid_full, verbose=False)

f_rat_xgb = xgb.XGBRegressor(**XGB_PARAMS);  f_rat_xgb.fit(X_full, rat_full, verbose=False)
f_rat_lgb = lgb.LGBMRegressor(**LGB_PARAMS);  f_rat_lgb.fit(X_full, rat_full)
f_rat_cat = CatBoostRegressor(**CAT_PARAMS);  f_rat_cat.fit(X_full, rat_full, verbose=False)

print("Full retrain complete.")

23:51:15 - cmdstanpy - INFO - Chain [1] start processing
23:51:16 - cmdstanpy - INFO - Chain [1] done processing


Full retrain complete.


#8. TEST FORCAST

In [107]:
comp_test  = extract_prophet_components(full_prophet, df_sub['Date'])
X_test     = _assemble_full_matrix(test_static, comp_test)
base_test  = comp_test['prophet_baseline']

test_log_xgb = base_test + f_rev_xgb.predict(X_test)
test_log_lgb = base_test + f_rev_lgb.predict(X_test)
test_log_cat = base_test + f_rev_cat.predict(X_test)

pred_rev_test = np.expm1(blend_rev.predict(
    np.column_stack([test_log_xgb, test_log_lgb, test_log_cat]))).clip(0)

pred_ratio_test = blend_rat.predict(
    np.column_stack([f_rat_xgb.predict(X_test),
                     f_rat_lgb.predict(X_test),
                     f_rat_cat.predict(X_test)])).clip(0, 0.99)

pred_cogs_test = np.minimum(pred_rev_test * pred_ratio_test, pred_rev_test * 0.99).clip(0)

print(f"Revenue : mean={pred_rev_test.mean():,.0f}  "
      f"min={pred_rev_test.min():,.0f}  max={pred_rev_test.max():,.0f}")
print(f"Ratio   : mean={pred_ratio_test.mean():.4f}  "
      f"min={pred_ratio_test.min():.4f}  max={pred_ratio_test.max():.4f}")


Revenue : mean=4,388,161  min=1,077,154  max=14,613,319
Ratio   : mean=0.8928  min=0.7982  max=0.9900


#9. EXPORT SUBMISSION

In [108]:
final_sub = pd.DataFrame({
    'Date'   : [d.strftime('%Y-%m-%d') for d in df_sub['Date']],
    'Revenue': np.round(pred_rev_test,  2),
    'COGS'   : np.round(pred_cogs_test, 2),
})
final_sub.to_csv('submission.csv', index=False)

assert (final_sub['Revenue'] >= 0).all(),              "Revenue < 0 detected"
assert (final_sub['COGS']    >= 0).all(),              "COGS < 0 detected"
assert (final_sub['COGS']    <= final_sub['Revenue']).all(), "COGS > Revenue detected"

print(f"\nsubmission.csv  ({len(final_sub)} rows)")
print(final_sub.head(5).to_string(index=False))


submission.csv  (548 rows)
      Date    Revenue       COGS
2023-01-01 2931059.13 2554044.79
2023-01-02 1627811.06 1403820.26
2023-01-03 1119581.36  909522.73
2023-01-04 1077154.08  859827.21
2023-01-05 1190405.70  960208.96


#SHAP

In [109]:
# ═══════════════════════════════════════════════════════════════════════════════
# SALESSTACK — MODEL VISUALIZATION  (save 6 separate PNG files)
# ───────────────────────────────────────────────────────────────────────────────
# Chạy ngay sau phần training ở trên.  Yêu cầu: shap, matplotlib.
# Output: salesstack_fig1.png … salesstack_fig6.png
# ═══════════════════════════════════════════════════════════════════════════════

import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
from sklearn.inspection import PartialDependenceDisplay

OUT_DIR  = ''          # thay bằng 'figures/' nếu muốn lưu vào thư mục riêng
OUT_FMT  = 'png'       # hoặc 'pdf', 'svg'
DPI      = 150

def out_path(n):
    return f"{OUT_DIR}salesstack_fig{n}.{OUT_FMT}"

# ── Shared aesthetics ────────────────────────────────────────────────────────
BLUE    = '#1d4ed8'
INDIGO  = '#4f46e5'
GRAY    = '#6b7280'
LGRAY   = '#f3f4f6'
RED     = '#dc2626'
AMBER   = '#d97706'
GREEN   = '#16a34a'
DARK    = '#111827'

CMAP_DIV = LinearSegmentedColormap.from_list(
    'salesstack', ['#1d4ed8', '#f9fafb', '#dc2626'])

plt.rcParams.update({
    'font.family'       : 'DejaVu Sans',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'axes.grid.axis'    : 'y',
    'grid.color'        : '#e5e7eb',
    'grid.linewidth'    : 0.6,
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : 'white',
    'font.size'         : 9,
})

# ── Feature-name alias map ───────────────────────────────────────────────────
FEAT_LABELS = {
    'month_progress'      : 'month_progress\n(tiến độ tháng)',
    'day'                 : 'day\n(ngày trong tháng)',
    't'                   : 't\n(thời gian tuyến tính)',
    'prophet_trend'       : 'prophet_trend\n(xu hướng)',
    'prophet_yearly'      : 'prophet_yearly\n(mùa vụ năm)',
    'peak_proximity'      : 'peak_proximity\n(gần đỉnh mùa vụ)',
    'pre_campaign_pulse'  : 'pre_campaign_pulse\n(chờ đợi τ=14)',
    'pre_campaign_sharp'  : 'pre_campaign_sharp\n(spike τ=7)',
    'post_campaign_trail' : 'post_campaign_trail\n(bão hoà τ=14)',
    'post_campaign_long'  : 'post_campaign_long\n(dư chấn τ=30)',
    'campaign_density_30' : 'campaign_density_30\n(mật độ 30 ngày)',
    'campaign_x_peak'     : 'campaign_x_peak\n(cộng hưởng)',
    'in_campaign'         : 'in_campaign\n(trong campaign)',
    'campaign_rank'       : 'campaign_rank\n(xếp hạng)',
    'prophet_uncert'      : 'prophet_uncert\n(rủi ro Prophet)',
    'promo_prob'          : 'promo_prob\n(xác suất promo)',
    'doy_norm'            : 'doy_norm\n(ngày trong năm)',
    'month_cos'           : 'month_cos\n(Fourier cosine)',
    'month_sin'           : 'month_sin\n(Fourier sine)',
    'year_parity'         : 'year_parity\n(năm lẻ/chẵn)',
    'is_peak_season'      : 'is_peak_season\n(đỉnh mùa vụ)',
}

def feat_label(f):
    return FEAT_LABELS.get(f, f)

def section_title(ax, text):
    ax.set_title(text, fontsize=11, fontweight='bold', color=DARK,
                 pad=10, loc='left')

# ── Compute SHAP values ──────────────────────────────────────────────────────
print("\nComputing SHAP values on full training data …")
_explainer   = shap.TreeExplainer(f_rev_xgb)
_shap_values = _explainer.shap_values(X_full)
_feat_names  = ALL_FEAT_COLS
_N_sample    = min(2000, len(X_full))
_rng         = np.random.default_rng(SEED)
_sample_idx  = _rng.choice(len(X_full), _N_sample, replace=False)
_Xs          = X_full[_sample_idx]
_SVs         = _shap_values[_sample_idx]

mean_abs_shap = np.abs(_shap_values).mean(axis=0)
top_idx       = np.argsort(mean_abs_shap)[::-1]
TOP_N         = 16

print(f"  SHAP done — {_N_sample} samples, {len(_feat_names)} features")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 1: Feature Importance comparison — XGB vs LGB vs CAT
# ─────────────────────────────────────────────────────────────────────────────
def plot_importance_comparison():
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    fig.suptitle('Feature Importance — So sánh giữa ba Booster (gain)',
                 fontsize=12, fontweight='bold', color=DARK, y=1.01)

    pairs = [
        (f_rev_xgb, 'XGBoost',  BLUE),
        (f_rev_lgb, 'LightGBM', INDIGO),
        (f_rev_cat, 'CatBoost', AMBER),
    ]

    for ax, (model, name, color) in zip(axes, pairs):
        try:
            if hasattr(model, 'feature_importances_'):
                imp = model.feature_importances_
            else:
                imp = model.get_feature_importance()

            imp_series = pd.Series(imp, index=_feat_names).nlargest(14)
            labels     = [feat_label(f).split('\n')[0] for f in imp_series.index]

            bars = ax.barh(range(len(imp_series)), imp_series.values,
                           color=color, alpha=0.82, edgecolor='none', height=0.65)
            for bar, val in zip(bars, imp_series.values):
                ax.text(bar.get_width() + imp_series.max() * 0.01,
                        bar.get_y() + bar.get_height() / 2,
                        f'{val:.3f}', va='center', fontsize=7.5, color=GRAY)

            ax.set_yticks(range(len(imp_series)))
            ax.set_yticklabels(labels, fontsize=8.5)
            ax.invert_yaxis()
            section_title(ax, name)
            ax.set_xlabel('Importance (gain / score)', fontsize=8.5)
            ax.set_xlim(0, imp_series.max() * 1.18)
        except Exception as e:
            ax.text(0.5, 0.5, f'N/A\n{e}', ha='center', va='center',
                    transform=ax.transAxes, color=GRAY)
            section_title(ax, name)

    plt.tight_layout()
    fig.savefig(out_path(1), dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f"  [1/6] Saved → {out_path(1)}")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 2: SHAP Beeswarm + Mean |SHAP| Bar
# ─────────────────────────────────────────────────────────────────────────────
def plot_shap_overview():
    fig = plt.figure(figsize=(15, 7))
    gs  = gridspec.GridSpec(1, 2, width_ratios=[1.4, 1], wspace=0.38)

    # LEFT: Mean |SHAP| bar
    ax_bar = fig.add_subplot(gs[1])
    top_feat = [_feat_names[i] for i in top_idx[:TOP_N]]
    top_shap = mean_abs_shap[top_idx[:TOP_N]]
    labels   = [feat_label(f).split('\n')[0] for f in top_feat]

    bars = ax_bar.barh(range(TOP_N), top_shap[::-1],
                       color=[BLUE if i < 3 else INDIGO if i < 8 else GRAY
                               for i in range(TOP_N)],
                       alpha=0.85, edgecolor='none', height=0.65)
    for bar, val in zip(bars, top_shap[::-1]):
        ax_bar.text(bar.get_width() + top_shap.max() * 0.01,
                    bar.get_y() + bar.get_height() / 2,
                    f'{val:.4f}', va='center', fontsize=7.5, color=GRAY)

    ax_bar.set_yticks(range(TOP_N))
    ax_bar.set_yticklabels(labels[::-1], fontsize=8.5)
    ax_bar.set_xlabel('Mean |SHAP value| (log-revenue scale)', fontsize=8.5)
    section_title(ax_bar, 'Feature Importance — Mean |SHAP|')

    # RIGHT: Beeswarm
    ax_bee = fig.add_subplot(gs[0])
    top_feat_bee = top_idx[:TOP_N]
    SVs_top = _SVs[:, top_feat_bee]
    Xs_top  = _Xs[:, top_feat_bee]
    Xs_norm = (Xs_top - Xs_top.min(axis=0)) / (np.ptp(Xs_top, axis=0) + 1e-9)

    rng = np.random.default_rng(0)
    for rank, feat_i in enumerate(top_feat_bee[::-1]):
        y_pos     = rank + rng.uniform(-0.28, 0.28, _N_sample)
        colors_pt = CMAP_DIV(Xs_norm[:, TOP_N - 1 - rank])
        ax_bee.scatter(_SVs[:, feat_i], y_pos,
                       c=colors_pt, s=5, alpha=0.55, linewidths=0, rasterized=True)

    ax_bee.axvline(0, color=DARK, linewidth=0.8, linestyle='--', alpha=0.5)
    ax_bee.set_yticks(range(TOP_N))
    ax_bee.set_yticklabels([feat_label(_feat_names[i]).split('\n')[0]
                            for i in top_feat_bee[::-1]], fontsize=8.5)
    ax_bee.set_xlabel('SHAP value  (tác động đến log Revenue)', fontsize=8.5)
    section_title(ax_bee, 'SHAP Beeswarm — Hướng & Độ lớn Tác động')

    sm = plt.cm.ScalarMappable(cmap=CMAP_DIV, norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax_bee, shrink=0.4, pad=0.01, aspect=20)
    cbar.set_ticks([0, 0.5, 1])
    cbar.set_ticklabels(['Low', 'Mid', 'High'], fontsize=7.5)
    cbar.set_label('Feature value\n(normalized)', fontsize=7.5)

    plt.suptitle('SHAP Analysis — XGBoost Revenue Model (full training data)',
                 fontsize=12, fontweight='bold', color=DARK, y=1.01)
    plt.tight_layout()
    fig.savefig(out_path(2), dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f"  [2/6] Saved → {out_path(2)}")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 3: SHAP Dependence Plots — top 6 features
# ─────────────────────────────────────────────────────────────────────────────
def plot_shap_dependence():
    top6  = [_feat_names[i] for i in top_idx[:6]]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes  = axes.flatten()

    for ax, feat in zip(axes, top6):
        fi   = _feat_names.index(feat)
        xval = _Xs[:, fi]
        yval = _SVs[:, fi]

        interact_scores = {
            j: np.corrcoef(xval, _Xs[:, j])[0, 1] ** 2
            for j in range(len(_feat_names)) if j != fi
        }
        best_interact = max(interact_scores, key=interact_scores.get)
        color_val     = _Xs[:, best_interact]
        color_norm    = (color_val - color_val.min()) / (np.ptp(color_val) + 1e-9)

        sc = ax.scatter(xval, yval, c=color_norm, cmap=CMAP_DIV,
                        s=10, alpha=0.55, linewidths=0, rasterized=True)

        sort_idx = np.argsort(xval)
        win      = max(1, len(xval) // 30)
        x_smooth = pd.Series(xval[sort_idx]).rolling(win, center=True).mean().values
        y_smooth = pd.Series(yval[sort_idx]).rolling(win, center=True).mean().values
        ax.plot(x_smooth, y_smooth, color=RED, linewidth=1.8,
                linestyle='--', label='Rolling mean', alpha=0.9)

        ax.axhline(0, color=DARK, linewidth=0.7, alpha=0.4)
        ax.set_xlabel(feat_label(feat), fontsize=8.5)
        ax.set_ylabel('SHAP value', fontsize=8.5)
        section_title(ax, f'Dependence: {feat.split("_")[0]}…')

        cbar = fig.colorbar(sc, ax=ax, shrink=0.7, pad=0.02, aspect=18)
        cbar.set_label(_feat_names[best_interact], fontsize=6.5)
        cbar.ax.tick_params(labelsize=6.5)

    plt.suptitle('SHAP Dependence Plots — Top 6 Features (màu sắc = đặc trưng tương tác)',
                 fontsize=12, fontweight='bold', color=DARK, y=1.01)
    plt.tight_layout()
    fig.savefig(out_path(3), dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f"  [3/6] Saved → {out_path(3)}")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 4: Partial Dependence Plots
# ─────────────────────────────────────────────────────────────────────────────
def plot_pdp():
    pdp_features = []
    priority = ['month_progress', 'day', 't', 'peak_proximity',
                'pre_campaign_pulse', 'campaign_density_30',
                'prophet_trend', 'campaign_rank']
    for f in priority:
        if f in _feat_names:
            pdp_features.append(_feat_names.index(f))
        if len(pdp_features) == 6:
            break

    if not pdp_features:
        print("  [4/6] Skipped — no matching features for PDP")
        return

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    PartialDependenceDisplay.from_estimator(
        f_rev_xgb, X_full, features=pdp_features,
        feature_names=_feat_names, ax=axes,
        line_kw={'color': BLUE, 'linewidth': 2},
        grid_resolution=50, percentiles=(0.05, 0.95),
        random_state=SEED,
    )

    for ax, fi in zip(axes, pdp_features):
        fname = _feat_names[fi]
        ax.set_xlabel(feat_label(fname), fontsize=8.5)
        ax.set_ylabel('Partial Dependence\n(log Revenue)', fontsize=8.5)
        section_title(ax, f'PDP: {fname}')
        ax.fill_between(ax.lines[0].get_xdata(), ax.lines[0].get_ydata(),
                        alpha=0.1, color=BLUE)

    plt.suptitle('Partial Dependence Plots — Tác động biên của từng đặc trưng đến log-Revenue',
                 fontsize=12, fontweight='bold', color=DARK, y=1.01)
    plt.tight_layout()
    fig.savefig(out_path(4), dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f"  [4/6] Saved → {out_path(4)}")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 5: Campaign Decay Kernel Visualization
# ─────────────────────────────────────────────────────────────────────────────
def plot_decay_kernels():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    days = np.linspace(0, 60, 300)

    # Pre-campaign
    ax = axes[0]
    for label, tau, color in [
        ('pre_campaign_sharp  (τ=7)',  7,  RED),
        ('pre_campaign_pulse  (τ=14)', 14, BLUE),
    ]:
        ax.plot(days, np.exp(-days / tau), color=color, linewidth=2.2, label=label)

    ax.axvspan(0, 7,  alpha=0.08, color=RED,  label='Spike zone (0–7 days)')
    ax.axvspan(7, 14, alpha=0.06, color=BLUE, label='Anticipation zone (7–14 days)')
    ax.set_xlabel('Số ngày trước khi campaign bắt đầu', fontsize=9.5)
    ax.set_ylabel('Signal strength  [0, 1]', fontsize=9.5)
    ax.legend(fontsize=8.5, framealpha=0.8)
    section_title(ax, 'Pre-Campaign Anticipation Kernels')
    ax.set_xlim(0, 60); ax.set_ylim(0, 1.05)
    ax.annotate('Tín hiệu đạt đỉnh\nngay trước sự kiện',
                xy=(0, 1), xytext=(10, 0.82),
                arrowprops=dict(arrowstyle='->', color=DARK, lw=1.2),
                fontsize=8, color=DARK)

    # Post-campaign
    ax = axes[1]
    for label, tau, color in [
        ('post_campaign_trail (τ=14)', 14, INDIGO),
        ('post_campaign_long  (τ=30)', 30, GREEN),
    ]:
        ax.plot(days, np.exp(-days / tau), color=color, linewidth=2.2, label=label)

    ax.axvspan(0,  14, alpha=0.07, color=INDIGO, label='Bão hoà mạnh (0–14 ngày)')
    ax.axvspan(14, 30, alpha=0.05, color=GREEN,  label='Dư chấn dài  (14–30 ngày)')
    ax.set_xlabel('Số ngày sau khi campaign kết thúc', fontsize=9.5)
    ax.set_ylabel('Signal strength  [0, 1]', fontsize=9.5)
    ax.legend(fontsize=8.5, framealpha=0.8)
    section_title(ax, 'Post-Campaign Hangover Kernels')
    ax.set_xlim(0, 60); ax.set_ylim(0, 1.05)
    ax.annotate('Dư chấn dài hạn\ncòn 22% sau 30 ngày',
                xy=(30, np.exp(-30/30)), xytext=(35, 0.5),
                arrowprops=dict(arrowstyle='->', color=DARK, lw=1.2),
                fontsize=8, color=DARK)

    plt.suptitle('Multi-Scale Exponential Decay Kernels — Campaign Feature Design',
                 fontsize=12, fontweight='bold', color=DARK)
    plt.tight_layout()
    fig.savefig(out_path(5), dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f"  [5/6] Saved → {out_path(5)}")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 6: OOF Diagnostics
# ─────────────────────────────────────────────────────────────────────────────
def plot_oof_diagnostics():
    fig = plt.figure(figsize=(15, 9))
    gs  = gridspec.GridSpec(2, 3, hspace=0.42, wspace=0.38)

    oof_actual    = np.expm1(oof_y_rev_log)
    oof_predicted = np.expm1(oof_blended)
    oof_residual  = oof_actual - oof_predicted

    # 1. Actual vs Predicted scatter
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.scatter(oof_actual, oof_predicted, alpha=0.35, s=5,
                color=BLUE, rasterized=True)
    lo = min(oof_actual.min(), oof_predicted.min())
    hi = max(oof_actual.max(), oof_predicted.max())
    ax1.plot([lo, hi], [lo, hi], color=RED, linewidth=1.5, linestyle='--', label='Lý tưởng')
    ax1.set_xlabel('Revenue thực tế', fontsize=8.5)
    ax1.set_ylabel('Revenue dự báo', fontsize=8.5)
    section_title(ax1, 'Actual vs Predicted (OOF)')
    ax1.legend(fontsize=8)

    mae_val  = np.abs(oof_residual).mean()
    rmse_val = np.sqrt((oof_residual ** 2).mean())
    r2_val   = 1 - np.sum(oof_residual**2) / np.sum((oof_actual - oof_actual.mean())**2)
    ax1.text(0.05, 0.95,
             f'MAE  = {mae_val:,.0f}\nRMSE = {rmse_val:,.0f}\nR²   = {r2_val:.4f}',
             transform=ax1.transAxes, fontsize=8, va='top',
             bbox=dict(boxstyle='round', facecolor=LGRAY, alpha=0.9))

    # 2. Residual vs Actual
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.scatter(oof_actual, oof_residual, alpha=0.35, s=5,
                color=INDIGO, rasterized=True)
    ax2.axhline(0, color=DARK, linewidth=1, linestyle='--', alpha=0.6)
    ax2.set_xlabel('Revenue thực tế', fontsize=8.5)
    ax2.set_ylabel('Residual (Actual − Predicted)', fontsize=8.5)
    section_title(ax2, 'Residual vs Actual')

    # 3. Residual distribution
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.hist(oof_residual, bins=60, color=AMBER, alpha=0.75, edgecolor='none')
    ax3.axvline(0, color=DARK, linewidth=1.5, linestyle='--')
    ax3.axvline(oof_residual.mean(), color=RED, linewidth=1.5,
                label=f'Mean = {oof_residual.mean():,.0f}')
    ax3.set_xlabel('Residual', fontsize=8.5)
    ax3.set_ylabel('Số quan sát', fontsize=8.5)
    section_title(ax3, 'Phân phối Residual')
    ax3.legend(fontsize=8)

    # 4. Time-series OOF (full bottom row)
    ax4 = fig.add_subplot(gs[1, :])
    n_oof      = len(oof_actual)
    x_idx      = np.arange(n_oof)
    fold_sizes = [365, 366, 365, 365]
    fold_edges = np.cumsum([0] + fold_sizes)[:5]

    ax4.fill_between(x_idx, oof_actual, alpha=0.3, color=GRAY, label='Actual')
    ax4.plot(x_idx, oof_actual,    color=GRAY, linewidth=0.6, alpha=0.5)
    ax4.plot(x_idx, oof_predicted, color=BLUE, linewidth=1.2, alpha=0.85,
             label='EnsembleBlender')

    for i, edge in enumerate(fold_edges[1:-1]):
        if edge < n_oof:
            ax4.axvline(edge, color=RED, linewidth=1, linestyle=':',
                        alpha=0.7, label='Fold split' if i == 0 else '')

    fold_labels  = ['2019', '2020', '2021', '2022']
    fold_centers = [(fold_edges[i] + fold_edges[i+1]) // 2 for i in range(4)]
    ylim_top     = ax4.get_ylim()[1]
    for lbl, cx in zip(fold_labels, fold_centers):
        if cx < n_oof:
            ax4.text(cx, ylim_top * 0.97 if ylim_top > 0 else 1,
                     lbl, ha='center', fontsize=8.5, color=RED, fontweight='bold')

    ax4.set_xlabel('Ngày (OOF, 2019–2022)', fontsize=8.5)
    ax4.set_ylabel('Revenue', fontsize=8.5)
    section_title(ax4, 'OOF Forecast vs Actual — Walk-Forward CV')
    ax4.legend(fontsize=8.5, loc='upper left')
    ax4.set_xlim(0, n_oof)

    plt.suptitle('OOF Diagnostics — EnsembleBlender Revenue (4-Fold Walk-Forward CV)',
                 fontsize=12, fontweight='bold', color=DARK)
    fig.savefig(out_path(6), dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f"  [6/6] Saved → {out_path(6)}")

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"STEP 7 — Generate 6 separate image files  (DPI={DPI}, fmt={OUT_FMT})")
print(f"{'='*65}")

plot_importance_comparison()
plot_shap_overview()
plot_shap_dependence()
plot_pdp()
plot_decay_kernels()
plot_oof_diagnostics()

print(f"\n✅ Done — 6 files saved:")
for i in range(1, 7):
    print(f"   {out_path(i)}")


Computing SHAP values on full training data …


  SHAP done — 2000 samples, 33 features

STEP 7 — Generate 6 separate image files  (DPI=150, fmt=png)
  [1/6] Saved → salesstack_fig1.png
  [2/6] Saved → salesstack_fig2.png
  [3/6] Saved → salesstack_fig3.png
  [4/6] Saved → salesstack_fig4.png
  [5/6] Saved → salesstack_fig5.png
  [6/6] Saved → salesstack_fig6.png

✅ Done — 6 files saved:
   salesstack_fig1.png
   salesstack_fig2.png
   salesstack_fig3.png
   salesstack_fig4.png
   salesstack_fig5.png
   salesstack_fig6.png
